# Melbourne Green-Space & Airbnb Pricing — Walkthrough

Does proximity to Melbourne's urban forest command a price premium on Airbnb?

This notebook orchestrates the `greenbnb` package end-to-end: it cleans the raw Inside Airbnb + urban-forest data, builds spatial tree-exposure features, adds review-text signals, and fits both an interpretable hedonic regression and a predictive gradient-boosted model.

> Runs on the bundled **synthetic** data by default. Place the real files in `data/raw/` to reproduce on actual Melbourne data — nothing below changes.

## 0. Setup

In [1]:
import sys, json
from pathlib import Path

# make the src/ package importable when running from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
from greenbnb import io, cleaning, geo, text, features, models, viz
from greenbnb.config import load_config
from greenbnb.io import raw_files_present
from greenbnb.synthetic import generate

cfg = load_config()
cfg.paths.ensure()
pd.set_option('display.width', 120)

## 1. Data

Use the real files if they are present; otherwise generate schema-faithful synthetic data so the notebook always runs.

In [2]:
present = raw_files_present(cfg)
if not (present.get('listings') and present.get('trees')):
    print('No real files found - generating synthetic dataset...')
    generate(cfg)
else:
    print('Using real raw files.')

listings = cleaning.clean_listings(io.load_listings(cfg), cfg)
trees    = cleaning.clean_trees(io.load_trees(cfg), cfg)
reviews  = cleaning.clean_reviews(io.load_reviews(cfg))
nbh      = io.load_neighbourhoods(cfg)
print(f'listings={len(listings):,}  trees={len(trees):,}  reviews={len(reviews):,}')
listings[['id','room_type','price','log_price','latitude','longitude']].head()

Using real raw files.


listings=2,500  trees=30,000  reviews=18,000


,id,room_type,price,log_price,latitude,longitude
0,1,Shared room,102.0,4.624973,-37.834555,144.922049
1,2,Entire home/apt,230.0,5.438079,-37.811906,144.978843
2,3,Private room,68.0,4.219508,-37.850928,144.974655
3,4,Entire home/apt,423.0,6.047372,-37.843869,144.953502
4,5,Private room,159.0,5.068904,-37.826129,144.969804


## 2. Exploratory views

Price is heavily right-skewed, so we model `log(price)`. Room type is the single largest price driver (we control for it throughout).

In [3]:
viz.plot_price_distribution(listings, cfg)
viz.plot_price_by_room_type(listings, cfg)
print('saved: price_distribution.png, price_by_room_type.png')
listings.groupby('room_type')['price'].agg(['count','median']).round(1)

saved: price_distribution.png, price_by_room_type.png


,count,median
room_type,,
Entire home/apt,1463,202.0
Hotel room,130,165.0
Private room,785,98.0
Shared room,122,57.5


## 3. Geospatial green-exposure features

Each listing is assigned to a neighbourhood polygon (point-in-polygon join), then we measure its exposure to the urban forest with a haversine `BallTree`:

- `trees_within_{100,250,500}m` — tree counts in each buffer
- `dist_nearest_tree_m` — distance to the closest tree
- `canopy_250m` — summed diameter-at-breast-height (canopy proxy)
- `genus_shannon_250m` — genus diversity within the primary buffer

In [4]:
listings = geo.assign_neighbourhood(listings, nbh, cfg)
listings = geo.green_exposure_features(listings, trees, cfg)
green_cols = [c for c in listings.columns if c.startswith(('trees_within','dist_nearest','canopy','genus_shannon'))]
listings[green_cols].describe().round(2)

,dist_nearest_tree_m,trees_within_100m,trees_within_250m,canopy_250m,genus_shannon_250m,trees_within_500m
count,2500.00,2500.00,2500.00,2500.00,2500.00,2500.00
mean,31.60,14.51,90.33,4084.88,1.87,354.84
std,20.57,17.73,100.34,4551.55,0.07,357.58
min,0.20,0.00,7.00,251.00,1.24,33.00
25%,15.80,4.00,30.00,1338.82,1.84,122.00
50%,27.50,7.00,43.00,1943.10,1.89,183.00
75%,43.60,17.00,109.00,4963.58,1.93,461.25
max,146.00,218.00,672.00,29651.80,1.94,1800.00


In [5]:
nb_density = geo.neighbourhood_tree_density(nbh, trees, cfg)
viz.plot_choropleth(nb_density, cfg, 'tree_density_km2')
viz.plot_green_vs_price(listings, cfg)
print('saved: choropleth_tree_density_km2.png, green_vs_price.png')
nb_density[['neighbourhood_geo' if 'neighbourhood_geo' in nb_density else 'neighbourhood','n_trees','tree_density_km2']].head()

saved: choropleth_tree_density_km2.png, green_vs_price.png


,neighbourhood,n_trees,tree_density_km2
0,Precinct A,774,125.3
1,Precinct B,2774,448.9
2,Precinct C,1259,203.7
3,Precinct D,546,88.4
4,Precinct E,1162,188.1


## 4. Review-text signals (NLP)

An **independent** green signal: how often guests volunteer green-amenity words (park, leafy, garden, ...), plus VADER sentiment. Aggregated to listing level.

In [6]:
reviews = text.add_green_score(text.add_sentiment(reviews), cfg)
review_agg = text.aggregate_to_listing(reviews)
print('Top review terms:')
print(text.top_terms(reviews, 12))
review_agg.head()

Top review terms:
apartment      2815
host           2815
book           2815
responsive     2815
clean          2815
central        2704
near           2704
station        2704
spot           2704
restaurants    2704
bed            2696
comfortable    2696
dtype: int64


,listing_id,n_reviews_text,mean_sentiment,mean_word_count,green_mention_rate,total_green_mentions
0,1,2,0.686000,7.500000,0.0,0
1,2,4,0.492850,8.500000,0.5,5
2,3,5,0.484220,8.600000,0.4,6
3,4,10,0.627540,9.100000,0.7,19
4,5,7,0.556514,7.428571,0.0,0


## 5. Model matrix + validity check

Merge everything. Then check whether the **spatial** green signal and the **textual** green signal agree — they should, if both track real greenery.

In [7]:
frame = features.build_model_frame(listings, review_agg, cfg)
r = cfg.geo.primary_radius_m
corr = frame['green_mention_rate'].corr(frame[f'trees_within_{r}m'])
print(f'spatial vs textual green correlation: {corr:.3f}')
frame.shape

spatial vs textual green correlation: 0.535


(2500, 33)

## 6. Hedonic regression — the green premium

`log(price) ~ green + controls + C(neighbourhood)` with HC3 robust SEs. Numeric predictors are standardised, so each coefficient is a per-SD effect; `exp(coef) - 1` converts it to an approximate percentage price change.

In [8]:
ols_num, ols_cat = features.ols_features(frame)
ols = models.hedonic_ols(frame, ols_num, ols_cat, cfg)
print(f'OLS R^2 = {ols.r2:.3f}  (n={ols.n:,})')
models.green_premium_summary(ols)

OLS R^2 = 0.579  (n=2,500)


,coef,p_value,pct_effect_per_sd,ci_low,ci_high
trees_within_250m,0.0334,0.0006,3.3987,0.0144,0.0524


### 6a. Robustness across green operationalisations

Each green measure enters its own regression (avoiding collinearity between nested buffers). A consistent positive, significant pattern is the validity argument for a real premium.

In [9]:
green_feats, controls, rcat = features.robustness_features(frame)
robustness = models.green_robustness(frame, green_feats, controls, rcat, cfg)
viz.plot_ols_green(robustness, cfg)
robustness

,green_feature,coef,p_value,pct_effect_per_sd,ci_low,ci_high,model_r2
0,trees_within_250m,0.0334,0.0006,3.40,0.0144,0.0524,0.5791
1,canopy_250m,0.0341,0.0004,3.47,0.0151,0.0530,0.5792
2,genus_shannon_250m,0.0197,0.0129,1.99,0.0042,0.0352,0.5781
3,dist_nearest_tree_m,-0.0116,0.1104,-1.16,-0.0259,0.0027,0.5774


## 7. Predictive model — what actually drives price?

A gradient-boosted regressor (all features, including every buffer) with permutation importance for an honest out-of-sample ranking.

In [10]:
X, y, num, cat = features.xy(frame, cfg)
ml = models.fit_price_model(X, y, num, cat, cfg)
print('metrics:', {k: round(v,3) for k,v in ml.metrics.items()})
viz.plot_importance(ml.importance, cfg)
ml.importance.head(10)

metrics: {'test_r2': 0.52, 'test_mae': 0.285, 'cv_r2_mean': 0.531, 'cv_r2_std': 0.021, 'n_train': 2000, 'n_test': 500}


,feature,importance,std,is_green
0,room_type,1.163314,0.072014,False
1,canopy_250m,0.018523,0.003869,True
2,trees_within_500m,0.017567,0.004873,True
3,dist_nearest_tree_m,0.012326,0.004949,True
4,genus_shannon_250m,0.004333,0.004303,True
5,mean_sentiment,0.002445,0.005900,False
6,trees_within_250m,0.001970,0.006020,True
7,calculated_host_listings_count,0.001428,0.002312,False
8,trees_within_100m,0.001364,0.004755,True
9,green_mention_rate,0.001317,0.003542,False


## 8. Interactive map

Listings coloured by tree exposure (green = high). Saved as an HTML artefact.

In [11]:
path = viz.interactive_map(frame, cfg)
print('saved interactive map ->', path.relative_to(cfg.paths.root))
# In Jupyter you can display it inline:
# from IPython.display import IFrame; IFrame(str(path), width='100%', height=500)

saved interactive map -> reports/figures/listings_green_exposure_map.html


## 9. Takeaways

- Listings with greater tree exposure carry a measurable price premium, consistent across tree count, canopy volume, and genus diversity.
- Distance to the nearest tree points the expected (negative) direction.
- The spatial and textual green signals agree, supporting construct validity.

**Caveats:** associational not causal; Inside Airbnb jitters coordinates (~150 m), attenuating fine-radius effects; the bundled run uses synthetic data with a clean embedded signal. Swap in the real files for genuine estimates.